# Phase 2 - Notebook 04: Gaussian Update Mechanism in SplaTAM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase2/04_gaussian_update.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand how new observations are fused into the Gaussian map via depth-guided initialization
2. Learn how to merge new Gaussians with existing ones during mapping
3. Implement joint optimization of Gaussian parameters and camera poses
4. Master geometric verification using depth consistency checks
5. Understand the Gaussian lifecycle: adding, updating, and deleting
6. Learn keyframe management strategies for online SLAM
7. Visualize the complete Gaussian update pipeline

**Estimated Time**: 90 minutes

**Prerequisites**: Phase 1 (3DGS fundamentals), Notebook 02 (SplaTAM Architecture), Notebook 03 (Map Initialization)

---

## 0. Environment Setup

配置环境和导入必要的库

In [ ]:
# Environment setup - 环境配置
import os
import sys

# Colab compatibility - Colab环境兼容
if 'COLAB_GPU' in os.environ:
    !pip install -q torch torchvision matplotlib numpy
    !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    %cd 3DGS-from-scratch

# Add project root to path - 添加项目根目录到路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse, FancyBboxPatch, FancyArrowPatch
from matplotlib.collections import PatchCollection
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

print("环境准备完成! Environment ready!")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Gaussian Update Overview

### 高斯更新机制概览

In SplaTAM, the Gaussian map is continuously updated as the camera explores the scene. This involves:

1. **New Observation Fusion**: Adding Gaussians from new depth observations
2. **Merging**: Combining overlapping Gaussians to maintain compact representation
3. **Joint Optimization**: Optimizing both Gaussians and poses simultaneously
4. **Geometric Verification**: Checking depth consistency to validate updates
5. **Lifecycle Management**: Adding, updating, and deleting Gaussians

```
SplaTAM Gaussian Update Pipeline:
┌─────────────────────────────────────────────────────────────────┐
│                    Gaussian Update Flow                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   RGB-D Input ──► Depth-Guided Initialization                    │
│                        │                                         │
│                        ▼                                         │
│              ┌─────────────────┐                                │
│              │  New Gaussians  │                                │
│              └────────┬────────┘                                │
│                       │                                         │
│            ┌─────────┴─────────┐                               │
│            ▼                   ▼                               │
│      ┌──────────┐       ┌──────────┐                          │
│      │ Merge    │       │ Add New  │                          │
│      │ Existing │       │ Regions  │                          │
│      └────┬─────┘       └────┬─────┘                          │
│           │                  │                                 │
│           └────────┬─────────┘                                 │
│                    ▼                                             │
│         ┌──────────────────┐                                  │
│         │ Joint Optimization│                                 │
│         │ (Gaussians + Poses)│                                │
│         └────────┬──────────┘                                  │
│                  │                                              │
│         ┌────────┴────────┐                                   │
│         ▼                 ▼                                   │
│   ┌──────────┐     ┌──────────┐                              │
│   │ Geometric│     │ Delete   │                              │
│   │ Verify   │     │ Unstable │                              │
│   └──────────┘     └──────────┘                              │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Visualize Gaussian update pipeline - 可视化高斯更新流程
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('SplaTAM Gaussian Update Pipeline', fontsize=16, fontweight='bold', pad=20)

# Define box style - 定义框样式
def draw_box(ax, x, y, w, h, text, color, fontsize=10):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1", 
                         facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', 
            fontsize=fontsize, fontweight='bold', wrap=True)

# Row 1: Input - 输入层
draw_box(ax, 0.5, 8.5, 2.5, 1, 'RGB-D\nInput', '#E3F2FD')

# Arrow - 箭头
ax.annotate('', xy=(4, 9), xytext=(3.2, 9),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Row 2: Initialization - 初始化
draw_box(ax, 4, 8.5, 3, 1, 'Depth-Guided\nInitialization', '#FFF9C4')

# Arrow down - 向下箭头
ax.annotate('', xy=(5.5, 7.5), xytext=(5.5, 8.4),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Row 3: New Gaussians - 新高斯
draw_box(ax, 4, 6.5, 3, 0.8, 'New Gaussians', '#C8E6C9')

# Branching arrows - 分支箭头
ax.annotate('', xy=(2.5, 5.8), xytext=(4.5, 6.5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(3.2, 6.3, 'Merge', fontsize=9, fontweight='bold', color='#2E7D32')

ax.annotate('', xy=(8.5, 5.8), xytext=(6.5, 6.5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(7.2, 6.3, 'New', fontsize=9, fontweight='bold', color='#1565C0')

# Row 4: Merge vs Add - 合并或添加
draw_box(ax, 1.5, 5, 2.5, 0.7, 'Merge Existing', '#A5D6A7')
draw_box(ax, 7.5, 5, 2.5, 0.7, 'Add New Regions', '#90CAF9')

# Arrows to optimization - 指向优化的箭头
ax.annotate('', xy=(5.5, 4.2), xytext=(2.8, 5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(5.5, 4.2), xytext=(8.8, 5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Row 5: Joint Optimization - 联合优化
draw_box(ax, 3.5, 3.2, 4, 0.9, 'Joint Optimization\n(Gaussians + Poses)', '#FFCCBC')

# Arrow down - 向下箭头
ax.annotate('', xy=(5.5, 2.4), xytext=(5.5, 3.1),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Row 6: Verification and Deletion - 验证和删除
draw_box(ax, 2, 1.5, 2.8, 0.8, 'Geometric\nVerification', '#FFCDD2')
draw_box(ax, 6.2, 1.5, 2.8, 0.8, 'Delete\nUnstable', '#D7CCC8')

# Feedback loop arrow - 反馈循环箭头
ax.annotate('', xy=(9.5, 3.6), xytext=(7.5, 2.3),
           arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2, 
                          connectionstyle="arc3,rad=0.3"))
ax.text(9.8, 3.0, 'Iterate', fontsize=9, fontweight='bold', color='#D32F2F')

# Output - 输出
ax.annotate('', xy=(12, 2.3), xytext=(9.2, 2.3),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))
draw_box(ax, 12, 1.8, 1.8, 1, 'Updated\nMap', '#E1BEE7')

# Legend - 图例
legend_y = 0.3
ax.text(1, legend_y, 'Colors:', fontsize=9, fontweight='bold')
colors = [('#E3F2FD', 'Input'), ('#FFF9C4', 'Init'), ('#C8E6C9', 'Gaussians'), 
          ('#FFCCBC', 'Optimization'), ('#E1BEE7', 'Output')]
x_pos = 2.5
for color, label in colors:
    rect = plt.Rectangle((x_pos, legend_y - 0.1), 0.3, 0.3, facecolor=color, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x_pos + 0.4, legend_y + 0.05, label, fontsize=8, va='center')
    x_pos += 1.8

plt.tight_layout()
plt.savefig('gaussian_update_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSplaTAM高斯更新流程包含三个主要阶段:")
print("1. 新观测融合 (New observation fusion)")
print("2. 联合优化 (Joint optimization)")
print("3. 生命周期管理 (Lifecycle management)")

## 2. Depth-Guided Gaussian Initialization

### 深度引导高斯初始化

When a new keyframe is selected, SplaTAM initializes new Gaussians from the depth map:

```python
# Pseudo-code: Depth-guided Gaussian initialization
def initialize_gaussians_from_depth(rgb, depth, K, pose):
    """
    从深度图初始化新高斯
    
    Args:
        rgb: [H, W, 3] RGB image
        depth: [H, W] depth map
        K: [3, 3] camera intrinsics
        pose: [4, 4] camera pose (world to camera)
    """
    h, w = depth.shape
    new_gaussians = []
    
    for v in range(h):
        for u in range(w):
            d = depth[v, u]
            if d > 0 and d < max_depth:  # Valid depth
                # 1. Back-project to 3D (反投影到3D)
                # P_camera = K^(-1) * [u, v, 1]^T * d
                P_cam = np.linalg.inv(K) @ np.array([u, v, 1]) * d
                
                # Transform to world coordinates (转换到世界坐标)
                P_world = (pose[:3, :3] @ P_cam + pose[:3, 3])
                
                # 2. Initialize Gaussian parameters (初始化高斯参数)
                gaussian = {
                    'xyz': P_world,           # 3D position
                    'rgb': rgb[v, u],         # Color from image
                    'scale': [k*d, k*d, epsilon],  # Scale based on depth
                    'rotation': [0, 0, 0, 1],  # Identity quaternion
                    'opacity': 0.5,           # Initial opacity
                    'view_count': 1           # Track visibility
                }
                new_gaussians.append(gaussian)
    
    return new_gaussians
```

**Key insights - 关键理解**:
- Each valid depth pixel creates one Gaussian
- Scale is proportional to depth (distant Gaussians are larger)
- Color is directly sampled from RGB image
- Initial opacity is set to 0.5 (uncertain)

In [ ]:
# Visualize depth-guided initialization - 可视化深度引导初始化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Simulate a scene - 模拟场景
np.random.seed(42)

# Create a simple depth map (simulating a room corner) - 创建简单深度图
h, w = 60, 80
depth = np.ones((h, w)) * 5.0  # Background depth

# Add a plane (wall) at z=2
depth[20:50, 30:70] = 2.0

# Add some noise - 添加噪声
depth += np.random.randn(h, w) * 0.05

# Create corresponding RGB - 创建对应RGB
rgb = np.zeros((h, w, 3))
rgb[20:50, 30:70] = [0.7, 0.5, 0.3]  # Wall color
rgb[:20, :] = [0.3, 0.3, 0.4]  # Ceiling
rgb[50:, :] = [0.4, 0.3, 0.3]  # Floor

# Panel 1: RGB Image - RGB图像
ax = axes[0, 0]
ax.imshow(rgb)
ax.set_title('RGB Image', fontsize=12, fontweight='bold')
ax.axis('off')

# Panel 2: Depth Map - 深度图
ax = axes[0, 1]
im = ax.imshow(depth, cmap='viridis')
ax.set_title('Depth Map', fontsize=12, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, label='Depth (m)')

# Panel 3: Sampling pattern - 采样模式
ax = axes[0, 2]
ax.imshow(rgb, alpha=0.3)
# Show sampling grid
sample_step = 5
for i in range(0, h, sample_step):
    for j in range(0, w, sample_step):
        if depth[i, j] < 4.9:  # Valid depth
            ax.plot(j, i, 'r.', markersize=3, alpha=0.5)
ax.set_title(f'Sampling Grid (step={sample_step})', fontsize=12, fontweight='bold')
ax.axis('off')

# Back-project to 3D - 反投影到3D
def back_project(depth, K, subsample=5):
    """Back-project depth to 3D points - 深度反投影到3D点"""
    h, w = depth.shape
    points = []
    colors = []
    
    K_inv = np.linalg.inv(K)
    
    for v in range(0, h, subsample):
        for u in range(0, w, subsample):
            d = depth[v, u]
            if d > 0.1 and d < 4.9:  # Valid depth
                # Back-project
                p_cam = K_inv @ np.array([u, v, 1]) * d
                points.append(p_cam)
                colors.append(rgb[v, u])
    
    return np.array(points), np.array(colors)

# Camera intrinsics (simplified) - 相机内参
K = np.array([[60, 0, 40],
              [0, 60, 30],
              [0, 0, 1]], dtype=float)

points, point_colors = back_project(depth, K, subsample=3)

# Panel 4: 3D Point Cloud - 3D点云
ax = axes[1, 0]
ax.scatter(points[:, 0], points[:, 2], c=point_colors, s=20, alpha=0.6)
ax.set_xlabel('X')
ax.set_ylabel('Z (Depth)')
ax.set_title('3D Point Cloud (Top View)', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Panel 5: Gaussian Visualization - 高斯可视化
ax = axes[1, 1]
# Visualize as ellipses - 用椭圆可视化
ax.scatter(points[:, 0], points[:, 2], c=point_colors, s=10, alpha=0.3)

# Add ellipses for some Gaussians - 为一些高斯添加椭圆
n_show = min(20, len(points))
indices = np.random.choice(len(points), n_show, replace=False)
for idx in indices:
    p = points[idx]
    # Scale proportional to depth
    scale = 0.1 + 0.05 * p[2]
    ellipse = Ellipse((p[0], p[2]), scale*2, scale*2, 
                      facecolor=point_colors[idx], alpha=0.4, edgecolor='black')
    ax.add_patch(ellipse)

ax.set_xlabel('X')
ax.set_ylabel('Z (Depth)')
ax.set_title('Gaussians (Ellipses)', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Panel 6: Statistics - 统计信息
ax = axes[1, 2]
ax.axis('off')

stats_text = f"""
Depth-Guided Initialization Statistics
深度引导初始化统计

Image Resolution: {w} x {h}
图像分辨率: {w} x {h}

Valid Depth Pixels: {len(points)}
有效深度像素: {len(points)}

New Gaussians Created: {len(points)}
创建的新高斯数: {len(points)}

Depth Range: [{depth.min():.2f}, {depth.max():.2f}] m
深度范围: [{depth.min():.2f}, {depth.max():.2f}] 米

Scale Formula: s = k * d
缩放公式: s = k * d
where k = 0.01, d = depth
其中 k = 0.01, d = 深度
"""

ax.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
        transform=ax.transAxes, verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('depth_guided_initialization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n深度引导初始化完成!")
print(f"从 {len(points)} 个有效深度像素创建了 {len(points)} 个新高斯")

## 3. Merging with Existing Gaussians

### 与现有高斯合并

New Gaussians may overlap with existing ones. SplaTAM uses spatial proximity to decide whether to merge or add:

```python
# Pseudo-code: Gaussian merging logic
def merge_gaussians(new_gaussians, existing_map, merge_threshold=0.05):
    """
    合并新高斯与现有地图
    
    Strategy:
    1. For each new Gaussian, check if it overlaps with existing ones
    2. If overlap is significant, update existing Gaussian
    3. Otherwise, add as new Gaussian
    """
    updated_gaussians = existing_map.copy()
    
    for new_g in new_gaussians:
        # Find nearby existing Gaussians
        nearby = find_nearby_gaussians(new_g['xyz'], existing_map, 
                                       threshold=merge_threshold)
        
        if len(nearby) > 0:
            # Merge with the closest one - 与最近的高斯合并
            closest = nearby[0]
            updated_gaussians[closest] = weighted_merge(
                updated_gaussians[closest], new_g
            )
        else:
            # Add as new Gaussian - 添加为新高斯
            updated_gaussians.append(new_g)
    
    return updated_gaussians

def weighted_merge(g1, g2):
    """Weighted average of two Gaussians - 两个高斯的加权平均"""
    # Use view_count as weight
    w1 = g1['view_count']
    w2 = g2['view_count']
    total = w1 + w2
    
    merged = {
        'xyz': (w1 * g1['xyz'] + w2 * g2['xyz']) / total,
        'rgb': (w1 * g1['rgb'] + w2 * g2['rgb']) / total,
        'view_count': total,
        # Update other parameters...
    }
    return merged
```

In [ ]:
# Visualize Gaussian merging - 可视化高斯合并
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Existing Gaussians - 现有高斯
np.random.seed(42)
existing = []
for i in range(8):
    g = {
        'xyz': np.array([np.random.uniform(-2, 2), np.random.uniform(0, 3)]),
        'color': plt.cm.viridis(i / 8),
        'radius': 0.3
    }
    existing.append(g)

# New Gaussians - 新高斯
new_gaussians = []
# Some overlapping, some new regions
new_positions = [
    ([0.5, 1.5], 'overlap'),  # Overlaps with existing
    ([-1.5, 0.5], 'overlap'),  # Overlaps with existing
    ([3.0, 2.0], 'new'),  # New region
    ([-3.0, 1.0], 'new'),  # New region
]

for pos, label in new_positions:
    g = {
        'xyz': np.array(pos),
        'color': '#FF6B6B',
        'radius': 0.25,
        'type': label
    }
    new_gaussians.append(g)

# Panel 1: Before merging - 合并前
ax = axes[0]

# Draw existing Gaussians - 绘制现有高斯
for g in existing:
    circle = plt.Circle(g['xyz'], g['radius'], color=g['color'], alpha=0.5,
                       edgecolor='black', linewidth=2)
    ax.add_patch(circle)
    ax.plot(g['xyz'][0], g['xyz'][1], 'ko', markersize=5)

# Draw new Gaussians - 绘制新高斯
for g in new_gaussians:
    circle = plt.Circle(g['xyz'], g['radius'], color=g['color'], alpha=0.7,
                       edgecolor='darkred', linewidth=2, linestyle='--')
    ax.add_patch(circle)
    ax.plot(g['xyz'][0], g['xyz'][1], 'rx', markersize=8, markeredgewidth=2)

ax.set_xlim(-4, 4)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Before Merging\n(合并前)', fontsize=13, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgray', edgecolor='black', label='Existing Gaussians'),
    Patch(facecolor='#FF6B6B', edgecolor='darkred', label='New Gaussians')
]
ax.legend(handles=legend_elements, loc='upper left')

# Panel 2: Merge decision visualization - 合并决策可视化
ax = axes[1]

# Draw existing Gaussians - 绘制现有高斯
for g in existing:
    circle = plt.Circle(g['xyz'], g['radius'], color=g['color'], alpha=0.3,
                       edgecolor='black', linewidth=1)
    ax.add_patch(circle)

# Show merge regions (dilated existing) - 显示合并区域
merge_threshold = 0.5
for g in existing:
    merge_circle = plt.Circle(g['xyz'], g['radius'] + merge_threshold, 
                             fill=False, edgecolor='blue', linewidth=2, 
                             linestyle=':', alpha=0.5)
    ax.add_patch(merge_circle)

# Draw new Gaussians with decision - 绘制带决策的新高斯
for g in new_gaussians:
    if g['type'] == 'overlap':
        marker = 'o'
        color = '#4ECDC4'
        label = 'Merge'
        # Draw merge arrow
        # Find closest existing
        closest = min(existing, key=lambda e: np.linalg.norm(e['xyz'] - g['xyz']))
        ax.annotate('', xy=closest['xyz'], xytext=g['xyz'],
                   arrowprops=dict(arrowstyle='->', color='#4ECDC4', lw=2))
    else:
        marker = 's'
        color = '#95E1D3'
        label = 'Add New'
    
    ax.scatter(g['xyz'][0], g['xyz'][1], marker=marker, s=200, 
              c=color, edgecolors='black', linewidth=2, zorder=5)

ax.set_xlim(-4, 4)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Merge Decision\n(合并决策)', fontsize=13, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Legend
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#4ECDC4', 
               markersize=12, label='Merge (合并)', markeredgecolor='black'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='#95E1D3', 
               markersize=12, label='Add New (添加)', markeredgecolor='black'),
    plt.Line2D([0], [0], linestyle=':', color='blue', linewidth=2, 
               label='Merge Threshold', alpha=0.5)
]
ax.legend(handles=legend_elements, loc='upper left')

# Panel 3: After merging - 合并后
ax = axes[2]

# Draw final map - 绘制最终地图
final_gaussians = existing.copy()

# Add non-overlapping new Gaussians - 添加不重叠的新高斯
for g in new_gaussians:
    if g['type'] == 'new':
        final_gaussians.append(g)

# Merge overlapping ones (visualized as combined circles) - 合并重叠的
for g in new_gaussians:
    if g['type'] == 'overlap':
        # Find closest and merge (simplified visualization)
        closest = min(existing, key=lambda e: np.linalg.norm(e['xyz'] - g['xyz']))
        merged_pos = (closest['xyz'] + g['xyz']) / 2
        merged_color = '#FFD93D'  # Yellow for merged
        circle = plt.Circle(merged_pos, 0.4, color=merged_color, alpha=0.7,
                           edgecolor='darkorange', linewidth=3)
        ax.add_patch(circle)
        ax.plot(merged_pos[0], merged_pos[1], '*', markersize=15, 
               color='darkorange', markeredgecolor='black', markeredgewidth=1)

# Draw non-merged existing - 绘制未合并的现有高斯
for i, g in enumerate(existing):
    # Check if this was merged
    was_merged = any(np.linalg.norm(g['xyz'] - ng['xyz']) < 1.0 
                    for ng in new_gaussians if ng['type'] == 'overlap')
    if not was_merged:
        circle = plt.Circle(g['xyz'], g['radius'], color=g['color'], alpha=0.5,
                           edgecolor='black', linewidth=2)
        ax.add_patch(circle)

# Draw new (non-overlapping) - 绘制新的(不重叠)
for g in new_gaussians:
    if g['type'] == 'new':
        circle = plt.Circle(g['xyz'], g['radius'], color='#95E1D3', alpha=0.7,
                           edgecolor='darkgreen', linewidth=2)
        ax.add_patch(circle)
        ax.plot(g['xyz'][0], g['xyz'][1], 'gs', markersize=10)

ax.set_xlim(-4, 4)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('After Merging\n(合并后)', fontsize=13, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Legend
legend_elements = [
    Patch(facecolor='lightgray', edgecolor='black', label='Unchanged'),
    Patch(facecolor='#FFD93D', edgecolor='darkorange', label='Merged'),
    Patch(facecolor='#95E1D3', edgecolor='darkgreen', label='Added New')
]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.savefig('gaussian_merging.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n高斯合并策略:")
print("1. Overlapping Gaussians → Merge (合并重叠)")
print("2. New regions → Add (添加新区域)")
print("3. Maintain compact representation (保持紧凑表示)")

## 4. Joint Optimization (Mapping)

### 联合优化（建图）

Unlike offline 3DGS where poses are fixed, SplaTAM jointly optimizes both Gaussian parameters AND camera poses during mapping:

```python
# Pseudo-code: Joint optimization in SplaTAM mapping
def joint_optimization(gaussian_map, keyframes, num_iters=100):
    """
    联合优化高斯地图和关键帧位姿
    
    Optimizes:
    - Gaussian parameters: {xyz, scale, rotation, opacity, SH}
    - Keyframe poses: {T_k} for each keyframe
    """
    # Collect parameters to optimize - 收集待优化参数
    params = []
    
    # Gaussian parameters - 高斯参数
    for g in gaussian_map.gaussians:
        g.xyz.requires_grad = True
        g.scale.requires_grad = True
        g.rotation.requires_grad = True
        g.opacity.requires_grad = True
        g.sh.requires_grad = True
        params.extend([g.xyz, g.scale, g.rotation, g.opacity, g.sh])
    
    # Keyframe poses - 关键帧位姿
    for kf in keyframes:
        kf.pose.requires_grad = True
        params.append(kf.pose)
    
    # Optimizer - 优化器
    optimizer = torch.optim.Adam(params, lr=0.001)
    
    for iter in range(num_iters):
        total_loss = 0
        
        # For each keyframe - 对每个关键帧
        for kf in keyframes:
            # Render from current pose estimate - 从当前位姿估计渲染
            rendered_rgb, rendered_depth = render(gaussian_map, kf.pose, kf.K)
            
            # Photometric loss - 光度损失
            loss_rgb = F.l1_loss(rendered_rgb, kf.rgb)
            
            # Depth loss - 深度损失
            loss_depth = F.l1_loss(rendered_depth, kf.depth)
            
            # Silhouette loss (optional) - 轮廓损失
            # loss_sil = compute_silhouette_loss(...)
            
            total_loss += loss_rgb + loss_depth
        
        # Backprop - 反向传播
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
    
    return gaussian_map, keyframes
```

**Key differences from offline 3DGS - 与离线3DGS的关键区别**:
1. **Pose optimization**: Camera poses are variables, not fixed
2. **Online learning**: Can't iterate over all frames infinitely
3. **Local window**: Only optimize nearby keyframes
4. **Catastrophic forgetting**: Must preserve old observations

In [ ]:
# Visualize joint optimization - 可视化联合优化
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Simulate optimization process - 模拟优化过程
np.random.seed(42)

# Ground truth - 真值
gt_pose = np.array([[1, 0, 0, 2],
                    [0, 1, 0, 0],
                    [0, 0, 1, 1],
                    [0, 0, 0, 1]], dtype=float)

# Initial estimate (with noise) - 初始估计
initial_pose = gt_pose.copy()
initial_pose[0, 3] += 0.5  # Offset in x
initial_pose[1, 3] -= 0.3  # Offset in y

# Gaussian map (simplified) - 高斯地图
gaussians = []
for i in range(5):
    g = {
        'xyz': np.array([np.random.uniform(-1, 3), np.random.uniform(-1, 2)]),
        'true_xyz': None,  # Will set later
    }
    g['true_xyz'] = g['xyz'].copy()
    g['xyz'] += np.random.randn(2) * 0.3  # Add noise
    gaussians.append(g)

# Panel 1: Initial state - 初始状态
ax = axes[0, 0]
# Draw Gaussians - 绘制高斯
for g in gaussians:
    ax.scatter(g['xyz'][0], g['xyz'][1], c='blue', s=200, alpha=0.6, zorder=3)
    ax.scatter(g['true_xyz'][0], g['true_xyz'][1], c='green', s=100, 
              alpha=0.6, marker='x', linewidths=3, zorder=4)

# Draw camera - 绘制相机
from matplotlib.patches import FancyArrowPatch
ax.scatter(initial_pose[0, 3], initial_pose[1, 3], c='red', s=300, 
          marker='^', edgecolors='black', linewidths=2, zorder=5)
ax.scatter(gt_pose[0, 3], gt_pose[1, 3], c='green', s=300, 
          marker='^', edgecolors='black', linewidths=2, zorder=5, alpha=0.5)

ax.set_xlim(-2, 4)
ax.set_ylim(-2, 3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Initial State\nIteration 0', fontsize=12, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Legend
ax.legend(['Gaussians (est.)', 'Gaussians (GT)', 'Camera (est.)', 'Camera (GT)'], 
         loc='upper left', fontsize=8)

# Panels 2-5: Optimization progress - 优化进度
iterations = [10, 30, 60, 100]

for idx, iters in enumerate(iterations):
    ax = axes[idx // 3, (idx % 3) + (1 if idx < 3 else 0)]
    
    # Simulate convergence - 模拟收敛
    alpha = min(1.0, iters / 100)
    
    # Interpolate pose - 插值位姿
    curr_pose = initial_pose.copy()
    curr_pose[0, 3] = initial_pose[0, 3] + alpha * (gt_pose[0, 3] - initial_pose[0, 3])
    curr_pose[1, 3] = initial_pose[1, 3] + alpha * (gt_pose[1, 3] - initial_pose[1, 3])
    
    # Interpolate Gaussians - 插值高斯
    for g in gaussians:
        curr_xyz = g['xyz'] + alpha * (g['true_xyz'] - g['xyz'])
        ax.scatter(curr_xyz[0], curr_xyz[1], c='blue', s=200, alpha=0.6, zorder=3)
        ax.scatter(g['true_xyz'][0], g['true_xyz'][1], c='green', s=100, 
                  alpha=0.6, marker='x', linewidths=3, zorder=4)
    
    # Draw camera
    ax.scatter(curr_pose[0, 3], curr_pose[1, 3], c='red', s=300, 
              marker='^', edgecolors='black', linewidths=2, zorder=5)
    ax.scatter(gt_pose[0, 3], gt_pose[1, 3], c='green', s=300, 
              marker='^', edgecolors='black', linewidths=2, zorder=5, alpha=0.5)
    
    # Draw error lines - 绘制误差线
    if idx == len(iterations) - 1:
        for g in gaussians:
            curr_xyz = g['true_xyz']
            ax.plot([g['xyz'][0], curr_xyz[0]], [g['xyz'][1], curr_xyz[1]], 
                   'b--', alpha=0.3, linewidth=1)
    
    ax.set_xlim(-2, 4)
    ax.set_ylim(-2, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Iteration {iters}', fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

# Panel 6: Loss curve - 损失曲线
ax = axes[1, 2]
iter_range = np.arange(0, 101)
# Simulated loss curves
loss_rgb = 0.5 * np.exp(-iter_range / 30) + 0.05
loss_depth = 0.3 * np.exp(-iter_range / 25) + 0.03
loss_total = loss_rgb + loss_depth

ax.plot(iter_range, loss_rgb, 'r-', linewidth=2, label='RGB Loss')
ax.plot(iter_range, loss_depth, 'b-', linewidth=2, label='Depth Loss')
ax.plot(iter_range, loss_total, 'g-', linewidth=2.5, label='Total Loss')

ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Joint Optimization Loss', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Annotate
ax.annotate('Both Gaussians\nand Poses\nconverge!', 
           xy=(80, loss_total[80]), xytext=(60, 0.4),
           arrowprops=dict(arrowstyle='->', color='green'),
           fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('joint_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n联合优化同时优化:")
print("1. 高斯参数 (位置、颜色、不透明度等)")
print("2. 相机位姿")
print("3. 通过RGB-D渲染损失进行监督")

## 5. Geometric Verification

### 几何验证

After optimization, SplaTAM performs geometric verification to ensure depth consistency:

```python
# Pseudo-code: Geometric verification
def geometric_verification(gaussian_map, keyframe):
    """
    几何验证：检查渲染深度与观测深度的一致性
    
    Returns:
        valid_mask: Boolean mask of valid Gaussians
    """
    # Render depth from current pose - 从当前位姿渲染深度
    rendered_depth = render_depth(gaussian_map, keyframe.pose, keyframe.K)
    
    # Compute depth error - 计算深度误差
    depth_error = torch.abs(rendered_depth - keyframe.depth)
    
    # Per-pixel verification - 逐像素验证
    valid_pixels = depth_error < threshold  # e.g., 0.05m
    
    # Back-project to Gaussians - 反投影到高斯
    for g in gaussian_map.gaussians:
        # Count how many valid pixels this Gaussian contributes to
        g.valid_count = count_valid_contributions(g, valid_pixels)
        
        # Update confidence - 更新置信度
        if g.valid_count > min_valid_threshold:
            g.confidence += 1
        else:
            g.confidence -= 1
    
    return gaussian_map
```

**Purpose - 目的**:
- Identify Gaussians that are consistent with depth observations
- Detect and mark unstable/outlier Gaussians
- Provide confidence scores for lifecycle management

In [ ]:
# Visualize geometric verification - 可视化几何验证
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

np.random.seed(42)
h, w = 60, 80

# Ground truth depth - 真值深度
gt_depth = np.ones((h, w)) * 3.0
gt_depth[20:40, 25:55] = 1.5

# Rendered depth (with some errors) - 渲染深度（带一些误差）
rendered_depth = gt_depth.copy()

# Add Gaussian in wrong place - 在错误位置添加高斯
rendered_depth[30:45, 40:60] = 2.0  # Wrong depth

# Add noise - 添加噪声
rendered_depth += np.random.randn(h, w) * 0.05

# Depth error - 深度误差
depth_error = np.abs(rendered_depth - gt_depth)

# Panel 1: Ground truth depth - 真值深度
ax = axes[0, 0]
im = ax.imshow(gt_depth, cmap='viridis', vmin=0, vmax=4)
ax.set_title('Ground Truth Depth', fontsize=13, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, label='Depth (m)')

# Panel 2: Rendered depth - 渲染深度
ax = axes[0, 1]
im = ax.imshow(rendered_depth, cmap='viridis', vmin=0, vmax=4)
ax.set_title('Rendered Depth', fontsize=13, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, label='Depth (m)')

# Highlight error region
error_mask = depth_error > 0.3
contours = ax.contour(error_mask, colors='red', linewidths=2)
ax.clabel(contours, inline=True, fontsize=8, fmt='Error')

# Panel 3: Depth error map - 深度误差图
ax = axes[1, 0]
im = ax.imshow(depth_error, cmap='hot', vmin=0, vmax=1)
ax.set_title('Depth Error |D_rendered - D_GT|', fontsize=13, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, label='Error (m)')

# Draw threshold line
threshold = 0.15
ax.contour(depth_error, levels=[threshold], colors='cyan', linewidths=3)
ax.text(70, 55, f'Threshold={threshold}m', color='cyan', fontsize=10, fontweight='bold')

# Panel 4: Validation result - 验证结果
ax = axes[1, 1]

# Valid regions - 有效区域
valid_mask = depth_error < threshold
invalid_mask = ~valid_mask

# Show valid/invalid
valid_img = np.zeros((h, w, 3))
valid_img[valid_mask] = [0.2, 0.8, 0.2]  # Green for valid
valid_img[invalid_mask] = [0.9, 0.2, 0.2]  # Red for invalid

ax.imshow(valid_img)
ax.set_title('Geometric Verification Result', fontsize=13, fontweight='bold')
ax.axis('off')

# Add statistics - 添加统计
valid_ratio = np.sum(valid_mask) / (h * w) * 100
stats_text = f"""
Geometric Verification Stats
几何验证统计

Threshold: {threshold}m
阈值: {threshold}米

Valid Pixels: {np.sum(valid_mask)} ({valid_ratio:.1f}%)
有效像素: {np.sum(valid_mask)} ({valid_ratio:.1f}%)

Invalid Pixels: {np.sum(invalid_mask)} ({100-valid_ratio:.1f}%)
无效像素: {np.sum(invalid_mask)} ({100-valid_ratio:.1f}%)
"""
ax.text(82, 30, stats_text, fontsize=10, family='monospace',
        verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=[0.2, 0.8, 0.2], label='Valid (有效)'),
    Patch(facecolor=[0.9, 0.2, 0.2], label='Invalid (无效)')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('geometric_verification.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n几何验证过程:")
print(f"1. 渲染深度图并与观测深度比较")
print(f"2. 计算逐像素深度误差")
print(f"3. 应用阈值: {threshold}m")
print(f"4. 标记有效/无效区域")
print(f"5. 更新高斯置信度")

## 6. Gaussian Lifecycle (Add/Update/Delete)

### 高斯生命周期（添加/更新/删除）

SplaTAM manages Gaussians through a complete lifecycle:

```python
# Pseudo-code: Gaussian lifecycle management
class GaussianLifecycleManager:
    """管理高斯生命周期的类"""
    
    def __init__(self):
        self.add_threshold = 0.5      # Min opacity to add
        self.delete_threshold = -3    # Min confidence before deletion
        self.min_views = 3            # Min views before stable
    
    def add_gaussians(self, new_candidates, existing_map):
        """Add new Gaussians after verification - 验证后添加新高斯"""
        for candidate in new_candidates:
            if candidate.opacity > self.add_threshold:
                candidate.age = 0
                candidate.confidence = 0
                existing_map.append(candidate)
    
    def update_gaussians(self, gaussian_map, keyframe):
        """Update existing Gaussians - 更新现有高斯"""
        for g in gaussian_map:
            # Check visibility in current frame - 检查当前帧可见性
            if is_visible(g, keyframe):
                g.view_count += 1
                g.last_seen = keyframe.id
                
                # Update parameters based on new observation
                g.confidence += 1 if verify_depth_consistency(g, keyframe) else -1
    
    def delete_unstable_gaussians(self, gaussian_map, current_frame_id):
        """Delete unstable Gaussians - 删除不稳定高斯"""
        to_delete = []
        
        for i, g in enumerate(gaussian_map):
            # Criteria for deletion - 删除标准
            if g.confidence < self.delete_threshold:
                to_delete.append(i)
            elif current_frame_id - g.last_seen > 50 and g.view_count < self.min_views:
                # Not seen for many frames with few views
                to_delete.append(i)
            elif g.opacity < 0.01:
                # Too transparent
                to_delete.append(i)
        
        # Remove in reverse order to maintain indices - 反向删除以保持索引
        for idx in reversed(to_delete):
            del gaussian_map[idx]
        
        return len(to_delete)
```

**Lifecycle states - 生命周期状态**:
1. **Newborn**: Just initialized, low confidence
2. **Growing**: Accumulating views, increasing confidence
3. **Stable**: Well-observed, high confidence
4. **Obsolete**: Not seen recently, candidate for deletion

In [ ]:
# Visualize Gaussian lifecycle - 可视化高斯生命周期
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

np.random.seed(42)

# Simulate Gaussian map evolution - 模拟高斯地图演化
frames = np.arange(0, 101, 10)

# Metrics over time - 随时间变化的指标
total_gaussians = []
stable_gaussians = []
deleted_gaussians = []
avg_confidence = []

for frame in frames:
    # Simulate growth and culling - 模拟增长和裁剪
    added = int(frame * 2.5)  # New Gaussians per frame
    culled = int(frame * 0.8) if frame > 20 else 0
    
    total = added - culled
    stable = int(total * 0.7) if frame > 30 else int(total * 0.3)
    
    total_gaussians.append(total)
    stable_gaussians.append(stable)
    deleted_gaussians.append(culled)
    avg_confidence.append(min(10, frame / 5))

# Panel 1: Gaussian count over time - 高斯数量随时间变化
ax = axes[0, 0]
ax.plot(frames, total_gaussians, 'b-', linewidth=2.5, label='Total', marker='o')
ax.plot(frames, stable_gaussians, 'g-', linewidth=2.5, label='Stable', marker='s')
ax.plot(frames, deleted_gaussians, 'r-', linewidth=2.5, label='Deleted', marker='^')

ax.set_xlabel('Frame Number', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Gaussian Count Over Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate
ax.axvline(x=30, color='gray', linestyle='--', alpha=0.5)
ax.text(32, max(total_gaussians) * 0.8, 'Stable phase\nbegins', fontsize=9)

# Panel 2: Confidence distribution - 置信度分布
ax = axes[0, 1]

# Show distribution at different frames
frame_indices = [3, 6, 10]  # Frames 30, 60, 100
colors = ['#FF9999', '#66B2FF', '#99FF99']

for idx, frame_idx in enumerate(frame_indices):
    frame_num = frames[frame_idx]
    # Simulate confidence distribution
    confidences = np.random.randn(100) * 2 + avg_confidence[frame_idx]
    confidences = np.clip(confidences, -5, 10)
    
    ax.hist(confidences, bins=20, alpha=0.6, label=f'Frame {frame_num}', 
            color=colors[idx], edgecolor='black')

ax.axvline(x=-3, color='red', linestyle='--', linewidth=2, label='Delete Threshold')
ax.set_xlabel('Confidence Score', fontsize=12)
ax.set_ylabel('Number of Gaussians', fontsize=12)
ax.set_title('Confidence Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: State transition diagram - 状态转换图
ax = axes[1, 0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Gaussian Lifecycle States', fontsize=13, fontweight='bold')

# State boxes - 状态框
states = [
    (2, 8, 'Newborn\n新生', '#FFE5B4'),
    (5, 8, 'Growing\n成长', '#B4D7FF'),
    (8, 8, 'Stable\n稳定', '#B4FFB4'),
    (5, 3, 'Obsolete\n过时', '#FFB4B4'),
]

for x, y, text, color in states:
    box = FancyBboxPatch((x-0.8, y-0.6), 1.6, 1.2, 
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=10, fontweight='bold')

# Arrows - 箭头
arrows = [
    ((2.8, 8), (4.2, 8), 'Observe', 'growing'),
    ((5.8, 8), (7.2, 8), 'Consistent', 'stable'),
    ((5, 7.2), (5, 3.8), 'Not seen', 'obsolete'),
    ((4.2, 3), (2.2, 7.2), 'Re-observed', 'revive'),
]

for start, end, label, style in arrows:
    color = '#2E7D32' if 'stable' in style or 'revive' in style else '#666'
    style_prop = "arc3,rad=0.2" if style == 'growing' else "arc3,rad=0"
    ax.annotate('', xy=end, xytext=start,
               arrowprops=dict(arrowstyle='->', color=color, lw=2,
                              connectionstyle=style_prop))
    mid_x = (start[0] + end[0]) / 2
    mid_y = (start[1] + end[1]) / 2
    ax.text(mid_x, mid_y + 0.3, label, fontsize=8, ha='center', color=color)

# Delete arrow - 删除箭头
ax.annotate('Delete', xy=(8, 1.5), xytext=(8, 7.2),
           arrowprops=dict(arrowstyle='->', color='red', lw=2.5),
           fontsize=10, color='red', fontweight='bold', ha='center')

ax.text(8, 1, 'Confidence < -3', fontsize=8, ha='center', color='red')

# Panel 4: Lifecycle statistics - 生命周期统计
ax = axes[1, 1]
ax.axis('off')

stats_text = """
Gaussian Lifecycle Statistics (at frame 100)
高斯生命周期统计（第100帧）

Total Gaussians: {total_gaussians[-1]}
总高斯数: {total_gaussians[-1]}

  • Newborn (0-2 views): {int(total_gaussians[-1] * 0.15)}
    新生 (0-2次观测)

  • Growing (3-9 views): {int(total_gaussians[-1] * 0.15)}
    成长 (3-9次观测)

  • Stable (10+ views): {stable_gaussians[-1]}
    稳定 (10+次观测)

Deleted Gaussians: {deleted_gaussians[-1]}
已删除高斯: {deleted_gaussians[-1]}

  • Low confidence: {int(deleted_gaussians[-1] * 0.6)}
    低置信度

  • Not observed: {int(deleted_gaussians[-1] * 0.3)}
    未被观测

  • Too transparent: {int(deleted_gaussians[-1] * 0.1)}
    过于透明
""".format(total_gaussians=total_gaussians, stable_gaussians=stable_gaussians, 
            deleted_gaussians=deleted_gaussians)

ax.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
        transform=ax.transAxes, verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('gaussian_lifecycle.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n高斯生命周期管理:")
print("• 添加: 初始化后验证通过的高斯")
print("• 更新: 根据观测次数和深度一致性更新置信度")
print("• 删除: 置信度低、长期未观测、过于透明的高斯")

## 7. Keyframe Management Strategy

### 关键帧管理策略

Keyframe selection is crucial for SLAM performance. SplaTAM uses multiple criteria:

```python
# Pseudo-code: Keyframe selection
def select_keyframe(current_frame, last_keyframe, gaussian_map):
    """
    关键帧选择策略
    
    Returns: True if current_frame should be a keyframe
    """
    # 1. Pose distance check - 位姿距离检查
    translation_dist = compute_translation_distance(
        current_frame.pose, last_keyframe.pose
    )
    rotation_dist = compute_rotation_distance(
        current_frame.pose, last_keyframe.pose
    )
    
    if translation_dist < min_translation and rotation_dist < min_rotation:
        return False  # Too similar, skip
    
    # 2. Covisibility check - 共视检查
    overlap_ratio = compute_covisibility_ratio(
        current_frame, last_keyframe, gaussian_map
    )
    
    if overlap_ratio > max_overlap:
        return False  # Too much overlap, redundant
    
    # 3. Information gain check - 信息增益检查
    new_region_ratio = compute_new_region_ratio(current_frame, gaussian_map)
    
    if new_region_ratio < min_new_ratio:
        return False  # Not enough new information
    
    return True  # Good keyframe candidate


def manage_keyframes(keyframe_list, max_keyframes=50):
    """Maintain compact keyframe list - 维护紧凑的关键帧列表"""
    if len(keyframe_list) <= max_keyframes:
        return keyframe_list
    
    # Culling strategy - 裁剪策略
    # 1. Remove redundant keyframes
    # 2. Keep keyframes with high covisibility diversity
    # 3. Prioritize recent keyframes
    
    scores = []
    for i, kf in enumerate(keyframe_list):
        score = kf.view_count * 0.5 + kf.age * 0.3 + kf.covisibility_diversity * 0.2
        scores.append((i, score))
    
    # Sort by score and keep top max_keyframes
    scores.sort(key=lambda x: x[1], reverse=True)
    to_keep = set(idx for idx, _ in scores[:max_keyframes])
    
    return [kf for i, kf in enumerate(keyframe_list) if i in to_keep]
```

**Keyframe selection criteria - 关键帧选择标准**:
1. **Motion threshold**: Sufficient camera movement
2. **Covisibility**: Not too much overlap with existing keyframes
3. **Novelty**: Sufficient new region coverage

In [ ]:
# Visualize keyframe management - 可视化关键帧管理
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

np.random.seed(42)

# Simulate camera trajectory - 模拟相机轨迹
n_frames = 100
t = np.linspace(0, 4*np.pi, n_frames)
trajectory = np.column_stack([
    np.cos(t) * 3,
    np.sin(t) * 3,
    np.ones(n_frames) * 1.5  # Height
])

# Keyframe selection - 关键帧选择
keyframes = [0]  # First frame is always keyframe
min_dist = 0.8  # Minimum distance between keyframes

for i in range(1, n_frames):
    last_kf = keyframes[-1]
    dist = np.linalg.norm(trajectory[i] - trajectory[last_kf])
    if dist > min_dist:
        keyframes.append(i)

# Panel 1: Trajectory with keyframes - 带关键帧的轨迹
ax = axes[0, 0]

# Draw full trajectory - 绘制完整轨迹
ax.plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=1, alpha=0.5, label='Trajectory')

# Draw all frames - 绘制所有帧
ax.scatter(trajectory[:, 0], trajectory[:, 1], c='lightblue', s=20, alpha=0.5)

# Draw keyframes - 绘制关键帧
kf_positions = trajectory[keyframes]
ax.scatter(kf_positions[:, 0], kf_positions[:, 1], c='red', s=100, 
          marker='s', edgecolors='black', linewidths=2, label='Keyframes', zorder=5)

# Number keyframes - 为关键帧编号
for i, idx in enumerate(keyframes[:10]):  # Show first 10
    ax.annotate(str(i+1), xy=trajectory[idx], xytext=(5, 5),
               textcoords='offset points', fontsize=9, fontweight='bold')

ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_title(f'Trajectory with Keyframes (n={len(keyframes)})', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Panel 2: Keyframe selection criteria - 关键帧选择标准
ax = axes[0, 1]

# Compute distances between consecutive frames - 计算连续帧之间的距离
distances = [np.linalg.norm(trajectory[i+1] - trajectory[i]) for i in range(n_frames-1)]

# Plot distances - 绘制距离
colors = ['red' if i in keyframes else 'blue' for i in range(n_frames-1)]
ax.scatter(range(n_frames-1), distances, c=colors, s=30, alpha=0.6)
ax.axhline(y=min_dist, color='red', linestyle='--', linewidth=2, label=f'Min Distance ({min_dist}m)')

ax.set_xlabel('Frame Number', fontsize=12)
ax.set_ylabel('Distance to Previous Frame (m)', fontsize=12)
ax.set_title('Keyframe Selection by Distance', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 3: Covisibility graph - 共视图
ax = axes[1, 0]

# Visualize keyframes as nodes with covisibility edges
n_kf = len(keyframes)
angles = np.linspace(0, 2*np.pi, n_kf, endpoint=False)
kf_x = np.cos(angles) * 2
kf_y = np.sin(angles) * 2

# Draw edges (covisibility) - 绘制边（共视关系）
for i in range(n_kf):
    for j in range(i+1, min(i+4, n_kf)):  # Connect to 3 neighbors
        # Line thickness based on covisibility
        covis = 1.0 - abs(i - j) / 4.0
        ax.plot([kf_x[i], kf_x[j]], [kf_y[i], kf_y[j]], 
               'gray', alpha=covis*0.5, linewidth=covis*3)

# Draw nodes - 绘制节点
sizes = np.random.uniform(100, 400, n_kf)
scatter = ax.scatter(kf_x, kf_y, s=sizes, c=range(n_kf), cmap='viridis', 
                    edgecolors='black', linewidths=2, zorder=5)

# Add labels - 添加标签
for i, (x, y) in enumerate(zip(kf_x, kf_y)):
    ax.text(x*1.3, y*1.3, f'KF{i}', ha='center', va='center', fontsize=9)

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Keyframe Covisibility Graph', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Keyframe ID')

# Panel 4: Keyframe management stats - 关键帧管理统计
ax = axes[1, 1]
ax.axis('off')

# Statistics - 统计
kf_ratio = len(keyframes) / n_frames * 100

stats_text = f"""
Keyframe Management Statistics
关键帧管理统计

Total Frames: {n_frames}
总帧数: {n_frames}

Keyframes Selected: {len(keyframes)}
选择的关键帧: {len(keyframes)}

Keyframe Ratio: {kf_ratio:.1f}%
关键帧比例: {kf_ratio:.1f}%

Selection Criteria:
选择标准:
  • Min Translation: {min_dist}m
    最小平移
  • Min Rotation: 10°
    最小旋转
  • Max Overlap: 80%
    最大重叠

Management Strategy:
管理策略:
  • Max Keyframes: 50
    最大关键帧数
  • Culling: Remove redundant
    裁剪: 删除冗余
  • Priority: View count + Age
    优先级: 观测次数+年龄
"""

ax.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
        transform=ax.transAxes, verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.7))

plt.tight_layout()
plt.savefig('keyframe_management.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n关键帧管理策略:")
print(f"1. 选择标准: 运动距离、共视重叠、信息增益")
print(f"2. 选择比例: {kf_ratio:.1f}% 的帧被选为关键帧")
print(f"3. 管理策略: 限制最大数量，删除冗余关键帧")

## 8. Summary and Key Takeaways

### 总结与关键要点

In this notebook, we covered the complete Gaussian update mechanism in SplaTAM:

| Component | Key Insight | Implementation |
|-----------|-------------|----------------|
| **Depth-Guided Init** | Each valid depth pixel → one Gaussian | Back-project using K, scale ∝ depth |
| **Merging** | Spatial proximity determines merge vs add | Distance threshold, weighted average |
| **Joint Optimization** | Optimize both Gaussians AND poses | RGB-D rendering loss, Adam optimizer |
| **Geometric Verification** | Depth consistency validates Gaussians | Per-pixel error threshold |
| **Lifecycle Management** | Gaussians evolve through states | Confidence tracking, culling |
| **Keyframe Management** | Select informative frames | Motion + covisibility + novelty |

### Core Principles - 核心原则

1. **Online Adaptation**: Unlike offline 3DGS, SLAM must process frames sequentially
2. **Compact Representation**: Merge redundant Gaussians, delete unstable ones
3. **Joint Optimization**: Poses and Gaussians are coupled and must be optimized together
4. **Verification**: Geometric checks ensure consistency with sensor data
5. **Selective Updates**: Keyframes focus computation on informative frames

### Next Steps - 下一步

**[05_camera_tracking.ipynb](./05_camera_tracking.ipynb)** - Dive deeper into camera tracking with render-and-compare

---

## References

1. SplaTAM: "Splat, Track & Map 3D Gaussians for Dense RGB-D SLAM" CVPR 2024
2. 3D Gaussian Splatting: "3D Gaussian Splatting for Real-Time Radiance Field Rendering" SIGGRAPH 2023
3. MonoGS: "Gaussian Splatting SLAM" CVPR 2024

In [ ]:
# Final visualization: Complete pipeline summary - 最终可视化：完整流程总结
print("\n" + "="*80)
print("SplaTAM Gaussian Update Mechanism - Complete Summary")
print("SplaTAM 高斯更新机制 - 完整总结")
print("="*80)

pipeline_summary = """

┌─────────────────────────────────────────────────────────────────────────┐
│                    GAUSSIAN UPDATE PIPELINE                              │
│                    高斯更新流程                                           │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  1. DEPTH-GUIDED INITIALIZATION                                          │
│     深度引导初始化                                                        │
│     • Back-project valid depth pixels to 3D                               │
│     • Initialize Gaussian: xyz, rgb, scale, rotation, opacity            │
│     • Scale proportional to depth (distant = larger)                     │
│                                                                          │
│  2. MERGING                                                              │
│     合并                                                                  │
│     • Check spatial proximity to existing Gaussians                      │
│     • If overlap: weighted merge                                         │
│     • If new region: add as new Gaussian                                 │
│                                                                          │
│  3. JOINT OPTIMIZATION (MAPPING)                                         │
│     联合优化（建图）                                                        │
│     • Optimize: Gaussian parameters + Camera poses                       │
│     • Loss: RGB-D rendering loss (L1)                                    │
│     • Local window: nearby keyframes only                                │
│                                                                          │
│  4. GEOMETRIC VERIFICATION                                               │
│     几何验证                                                              │
│     • Compare rendered depth with observed depth                         │
│     • Per-pixel error thresholding                                       │
│     • Update Gaussian confidence scores                                  │
│                                                                          │
│  5. LIFECYCLE MANAGEMENT                                                 │
│     生命周期管理                                                          │
│     • Add: High-opacity, verified Gaussians                              │
│     • Update: Confidence based on consistency                            │
│     • Delete: Low confidence, long-unseen, transparent                   │
│                                                                          │
│  6. KEYFRAME MANAGEMENT                                                  │
│     关键帧管理                                                            │
│     • Select: Motion threshold + covisibility + novelty                  │
│     • Maintain: Max 50 keyframes, cull redundant                         │
│     • Optimize: Local window around current frame                        │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘

Key Differences from Offline 3DGS:
与离线3DGS的关键区别:
• Online: Process frames sequentially (not batch)                          
  在线: 顺序处理帧（非批量）                                                
• Joint: Optimize poses AND Gaussians (not just Gaussians)                 
  联合: 同时优化位姿和高斯（而非仅高斯）                                      
• Compact: Merge and delete to control map size                            
  紧凑: 合并和删除以控制地图大小                                             
• Selective: Keyframe-based optimization                                   
  选择性: 基于关键帧的优化                                                   
"""

print(pipeline_summary)
print("\n" + "="*80)
print("End of Notebook 04: Gaussian Update Mechanism")
print("笔记本04结束: 高斯更新机制")
print("="*80)